In [ ]:
# Load dataset
from datasets import load_dataset

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
ds

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

# Część 1

### 1.1

In [ ]:
import re
import unicodedata
from collections import Counter


UNK_TOKEN = '<UNK>'
TOKEN_PATTERN = re.compile(r"[^\W_]+(?:'[^\W_]+)*")


def normalize_text(text):
    return unicodedata.normalize('NFKC', text).casefold()


def tokenize(text):
    return TOKEN_PATTERN.findall(normalize_text(text))

# Unknown tokens are saved as the UNK_TOKEN in the vocabulary
def build_vocabulary(training_texts):
    tokens = {token for text in training_texts for token in tokenize(text)}
    vocabulary = {UNK_TOKEN: 0}
    vocabulary.update({token: index for index, token in enumerate(sorted(tokens), start=1)})
    return vocabulary

In [ ]:
train_texts = ds['train']['text']
vocabulary = build_vocabulary(train_texts)

vocabulary

{'<UNK>': 0,
 '00': 1,
 '007': 2,
 '1': 3,
 '10': 4,
 '100': 5,
 '101': 6,
 '102': 7,
 '104': 8,
 '105': 9,
 '10th': 10,
 '11': 11,
 '110': 12,
 '112': 13,
 '117': 14,
 '11th': 15,
 '12': 16,
 '123': 17,
 '125': 18,
 '127': 19,
 '129': 20,
 '12th': 21,
 '13': 22,
 '13th': 23,
 '14': 24,
 '146': 25,
 '15': 26,
 '15th': 27,
 '16': 28,
 '163': 29,
 '17': 30,
 '170': 31,
 '179': 32,
 "1790's": 33,
 '18': 34,
 '180': 35,
 '1873': 36,
 '1899': 37,
 '18th': 38,
 '19': 39,
 '1915': 40,
 "1920's": 41,
 '1930s': 42,
 '1933': 43,
 '1934': 44,
 '1940s': 45,
 '1949': 46,
 '1950': 47,
 "1950's": 48,
 '1950s': 49,
 '1952': 50,
 '1953': 51,
 '1954': 52,
 '1955': 53,
 '1957': 54,
 '1958': 55,
 '1960': 56,
 '1960s': 57,
 '1962': 58,
 '1967': 59,
 '1970s': 60,
 '1972': 61,
 '1973': 62,
 '1975': 63,
 '1978': 64,
 '1979': 65,
 '1980': 66,
 "1980's": 67,
 '1980s': 68,
 '1982': 69,
 "1982's": 70,
 "1983's": 71,
 '1984': 72,
 '1986': 73,
 "1988's": 74,
 '1989': 75,
 '1990': 76,
 '1991': 77,
 '1992': 78,
 "199

### 1.2

In [ ]:
def bag_of_words(text, vocabulary):
    vector = {}
    for token in tokenize(text):
        index = vocabulary.get(token, vocabulary[UNK_TOKEN])
        vector[index] = vector.get(index, 0) + 1
    return vector

In [11]:
import random

random.seed(42)

rand_train_texts = random.sample(train_texts, 5)
for text in rand_train_texts:
    bow_vector = bag_of_words(text, vocabulary)
    print(text, bow_vector)

a touching , sophisticated film that almost seems like a documentary in the way it captures an italian immigrant family on the brink of major changes . {173: 2, 15474: 1, 14033: 1, 5737: 1, 15182: 1, 587: 1, 13258: 1, 8824: 1, 4393: 1, 7613: 1, 15186: 2, 16581: 1, 8103: 1, 2263: 1, 682: 1, 8107: 1, 7531: 1, 5523: 1, 10461: 1, 1954: 1, 10405: 1, 9158: 1, 2449: 1}
starts off with a bang , but then fizzles like a wet stick of dynamite at the very end . it's still worth a look . {14340: 1, 10406: 1, 16840: 1, 173: 3, 1271: 1, 2110: 1, 15204: 1, 5813: 1, 8824: 1, 16676: 1, 14403: 1, 10405: 1, 4673: 1, 1042: 1, 15186: 1, 16323: 1, 4940: 1, 8106: 1, 14414: 1, 16932: 1, 8961: 1}
sometimes there are very , very good reasons for certain movies to be sealed in a jar and left on a remote shelf indefinitely . {14013: 1, 15212: 1, 880: 1, 16323: 2, 6509: 1, 12154: 1, 5960: 1, 2414: 1, 9878: 1, 15386: 1, 1360: 1, 13206: 1, 7613: 1, 173: 2, 8171: 1, 701: 1, 8703: 1, 10461: 1, 12409: 1, 13488: 1, 7672:

### 1.3

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader

torch.manual_seed(42)


def bow_batch(batch):
    # Build dense vectors only for the current batch.
    features = torch.zeros((len(batch), len(vocabulary)), dtype=torch.float32)
    for row, (vector, label) in enumerate(batch):
        for index, count in vector.items():
            features[row, index] = count
    labels = torch.tensor([label for vector, label in batch], dtype=torch.float32)
    return features, labels.unsqueeze(1)


training_data = [
    (bag_of_words(text, vocabulary), label)
    for text, label in zip(train_texts, ds['train']['label'])
]
train_loader = DataLoader(
    training_data, batch_size=64, shuffle=True, collate_fn=bow_batch
)

# One linear layer returns one logit per review (no hidden layers).
model = nn.Linear(len(vocabulary), 1)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in range(10):
    total_loss = 0.0
    for features, labels in train_loader:
        optimizer.zero_grad()
        logits = model(features)
        probabilities = torch.sigmoid(logits)
        loss = criterion(probabilities, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    print(f'Epoch {epoch + 1:2d}: BCE = {total_loss / len(training_data):.4f}')


def predict_sentiment(text):
    model.eval()
    features, _ = bow_batch([(bag_of_words(text, vocabulary), 0)])
    with torch.no_grad():
        probability = torch.sigmoid(model(features)).item()
    return {'probability': probability, 'label': int(probability >= 0.5)}


predict_sentiment('A wonderful movie with great acting.')